# Tech Challenge Fase 2
## 04 — Streaming Orquestrador

Prepara a estrutura da pipeline híbrida:

```text
streaming/
├── input/
├── bronze_streaming/
├── silver_streaming/
├── gold_streaming/
└── checkpoint/
```

Ordem:
`04 → 04_1 → 04_2 → 04_3 → 04_4`

## 1. Imports

In [0]:
import json
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

## 2. Configuração

In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"
config = json.loads(dbutils.fs.head(CONFIG_FILE_PATH))

STREAMING_PATH = config["paths"]["streaming_path"]
LOG_PATH = config["paths"]["log_path"]
CONFIG_PATH = config["paths"]["config_path"]
EXECUTION_DATE = config["project"]["execution_date"]

INPUT_PATH = f"{STREAMING_PATH}/input"
BRONZE_PATH = f"{STREAMING_PATH}/bronze_streaming"
SILVER_PATH = f"{STREAMING_PATH}/silver_streaming"
GOLD_PATH = f"{STREAMING_PATH}/gold_streaming"
CHECKPOINT_BRONZE = f"{STREAMING_PATH}/checkpoint/bronze"
CHECKPOINT_SILVER = f"{STREAMING_PATH}/checkpoint/silver"
CHECKPOINT_GOLD = f"{STREAMING_PATH}/checkpoint/gold"

print("BRONZE_PATH:", BRONZE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("LOG_PATH:", LOG_PATH)
print("CONFIG_PATH:", CONFIG_PATH)
print("EXECUTION_DATE:", EXECUTION_DATE)

## 3. Criação dos diretórios

In [0]:
for path in [
    INPUT_PATH, BRONZE_PATH, SILVER_PATH, GOLD_PATH,
    CHECKPOINT_BRONZE, CHECKPOINT_SILVER, CHECKPOINT_GOLD,
    f"{LOG_PATH}/pipeline_execution/streaming",
    f"{LOG_PATH}/data_quality/streaming",
    f"{LOG_PATH}/rejected/streaming"
]:
    dbutils.fs.mkdirs(path)

print("Estrutura Streaming criada/validada.")

## 4. Schema oficial dos eventos

In [0]:
event_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("event_timestamp", TimestampType(), False),
    StructField("event_type", StringType(), False),
    StructField("ano", IntegerType(), False),
    StructField("co_uf", StringType(), True),
    StructField("sg_uf", StringType(), True),
    StructField("co_municipio", StringType(), True),
    StructField("no_municipio", StringType(), True),
    StructField("indicador", StringType(), False),
    StructField("valor", DoubleType(), True),
    StructField("origem", StringType(), True),
    StructField("event_version", IntegerType(), False)
])
print(event_schema.simpleString())

## 5. Streaming metadata

In [0]:
streaming_metadata = {
    "input_path": INPUT_PATH,
    "bronze_path": BRONZE_PATH,
    "silver_path": SILVER_PATH,
    "gold_path": GOLD_PATH,
    "checkpoint_bronze": CHECKPOINT_BRONZE,
    "checkpoint_silver": CHECKPOINT_SILVER,
    "checkpoint_gold": CHECKPOINT_GOLD,
    "event_types": ["indicador_atualizado","meta_atualizada","resultado_corrigido"],
    "valid_value_range": {"min": 0.0, "max": 100.0},
    "execution_date": EXECUTION_DATE
}

dbutils.fs.put(
    f"{CONFIG_PATH}/streaming_metadata.json",
    json.dumps(streaming_metadata, indent=4, ensure_ascii=False),
    overwrite=True
)

print("Streaming Planejador concluído.")